# 05. Lecke - Ügynöki RAG


## Beállítás

Ez a jegyzetfüzet bemutatja az Agentic RAG (Retrieval-Augmented Generation) mintát a Microsoft Agent Framework használatával.

**Előfeltételek:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — az Azure AI Search szolgáltatásod végpontja
- `AZURE_SEARCH_API_KEY` — az Azure AI Search API kulcsod
- Azure OpenAI telepítés környezeti változókon keresztül konfigurálva
- Azure CLI hitelesítve (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mi az az Agentic RAG?

A hagyományos RAG egy rögzített folyamatot követ: először dokumentumokat keres, majd választ generál. Az **Agentic RAG** továbbmegy azzal, hogy az ügynök autonómiát kap arra, hogy eldöntse, **mikor** és **hogyan** szerezze be az információt.

Az Agentic RAG használatával az ügynök képes:
- **Dönteni** arról, hogy szükséges-e lekérdezés a kérdés megválaszolása előtt
- **Kiválasztani**, mely adatforrást vagy eszközt kérdezze le
- **Értékelni** a lekérdezett eredményeket, és ha az első próbálkozás nem elégséges, további lekérdezéseket végezni
- **Összevonni** a több lekérdezési lépésből származó információkat egy koherens válaszba

Ez az ügynök rugalmasabbá és pontosabbá tételét eredményezi a statikus lekérdezés- majd generálás folyamatával szemben.


## Keresőeszköz létrehozása

Az Agentic RAG-ben a külső adatforrások **eszközökként** vannak becsomagolva, amelyeket az ügynök igény szerint meghívhat. Ez lehetővé teszi az ügynök számára, hogy a lekérést egyszerűen egy másik műveletként kezelje, nem pedig kötelező lépésként.

Lent egy utazási tudásbázist definiálunk, és eszközként tesszük elérhetővé, hogy az ügynök célállomás-információkat keresve hívhassa meg.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## A RAG ügynök létrehozása

Most létrehozunk egy ügynököt, akit arra utasítanak, hogy **mindig információt szerezzen be a válaszadás előtt**. Az ügynök a `search_travel_knowledge` eszközt használja, hogy válaszait a tudásbázison alapozza meg, ahelyett, hogy a saját tanítási adatára támaszkodna.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iteratív lekérdezés — A Maker-Checker minta

Az Agentic RAG egyik fő előnye az **iteratív lekérdezés**. Az ügynök többszörös keresési fordulókat végezhet az első eredményeinek ellenőrzésére, finomítására vagy bővítésére — hasonlóan a "maker-checker" munkafolyamathoz:

1. **Maker lépés**: Az ügynök lekéri az alapinformációkat és megalkot egy választ.
2. **Checker lépés**: Az ügynök további lekérdezéseket végez a részletek ellenőrzésére vagy hiányok pótlására.

Az alábbiakban az ügynök egy olyan kérdést kap, amely több desztináció összehasonlítását igényli, ezáltal többszöri keresésre ösztönzi.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Összefoglalás

Ebben a leckében megtanultad, hogyan építs fel egy **Agentic RAG** rendszert a Microsoft Agent Framework használatával:

- Az **Agentic RAG** lehetővé teszi, hogy az ügynökök önállóan döntsenek arról, mikor kérjenek le információt, így a lekérés dinamikus lesz, nem pedig fix.
- **Eszközök mint adatforrások**: Külső tudásbázisok (például Azure AI Search) eszközként vannak becsomagolva, amelyeket az ügynök meghívhat.
- **Ismétlődő lekérés**: A maker-checker mintázat lehetővé teszi, hogy az ügynök többször is lekérdezzen — keresve, ellenőrizve és finomítva — mielőtt végső válasz készül.

Éles környezetben a memóriában lévő `TRAVEL_KNOWLEDGE_BASE` helyett valódi Azure AI Search indexet használnál a nagy léptékű utazási dokumentumok lekéréséhez.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
